# Etapa 3 · Recuperación: primera línea de base BM25

**Objetivo:** medir qué evidencia entrega BM25 sobre el corpus jurídico antes de generar respuestas.
Esta versión inicial no completa la grilla de ablaciones: deja una base para sumar denso y RRF.
No requiere Ollama ni GPU; descarga únicamente dos tokenizers abiertos.

**Entradas:** `artifacts/full/documents.jsonl`, módulos del proyecto y evaluación congelada de etapa 2.
**Salidas:** índice/chunks/manifiesto en `artifacts/<profile>/index/stage3_<hash>/`;
resultados por pregunta, contexto entregado, tablas e informe de ejecución en `results/stage3_<hash>/`.

El notebook contiene las funciones necesarias para poder subirse solo a Colab. Se sincronizan desde
`retrieval_stage3.py` mediante `build_stage3_notebook.py`; el bootstrap conserva el de las etapas anteriores.
No modifica ni vuelve a congelar la evaluación. No hay resultados reales precargados.

**Antes de interpretar resultados:** completar la revisión humana pendiente de la etapa 2.
Las salidas smoke sirven para depurar; los resultados dev sirven para desarrollar. Solo test sirve para el reporte final.

In [ ]:
# --- Bootstrap de Colab (no hace nada fuera de Colab) --------------------------------
# Colab arranca en /content con un entorno vacío: sin esta celda, el `import environment`
# siguiente falla con ModuleNotFoundError. La genera tools/py_to_notebook.py -- editala
# ahí, no acá.
import hashlib
import os
import subprocess
import sys

# El disco de Colab se borra cuando se desconecta el entorno de ejecución (~90 min inactivo,
# 12 h como máximo). Con USE_DRIVE = True, artifacts/ (documentos, embeddings, índices,
# resultados, cachés del LLM) vive en cambio en tu Google Drive, y al volver a correr se
# saltea todo lo que ya está construido.
USE_DRIVE = True
# Guardar también en Drive la caché de modelos de HuggingFace (~3 GB). Apagado por defecto:
# volver a descargar suele ser más rápido que leer archivos grandes a través del montaje de Drive.
MODELS_ON_DRIVE = True

# Un entorno de ejecución de Colab, desde el navegador o desde la extensión de Colab de VS Code
# (cuyo kernel puede no haber importado google.colab: la propia variable de entorno del runtime
# dice dónde estamos).
if "google.colab" in sys.modules or os.environ.get("COLAB_RELEASE_TAG"):
    REPO_URL = "https://github.com/Jahdejosefina/projecto_ragTP2.git"
    REPO_BRANCH = ""
    REPO_DIR = "projecto_rag"
    DRIVE = "/content/drive/MyDrive/projecto_rag"
    quiet = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}  # nunca esperar un pedido de contraseña

    def github_token():
        # Un repositorio privado: un token de GitHub como secreto de Colab llamado GITHUB_TOKEN.
        # Los secretos de Colab solo se pueden leer desde el navegador; una variable de entorno
        # funciona en cualquier lado.
        token = os.environ.get("GITHUB_TOKEN")
        if not token:
            try:
                from google.colab import userdata
                token = userdata.get("GITHUB_TOKEN")
            except Exception:
                token = None
        return token

    # Dónde ESTAMOS antes que qué hay en disco: `isdir(REPO_DIR)` es relativo, así que
    # chequearlo primero clonaría una copia anidada cada vez que se vuelve a correr esta celda
    # en la misma sesión.
    if os.path.basename(os.getcwd()) != REPO_DIR:
        if not os.path.isdir(REPO_DIR):
            cmd = ["git", "clone", "--depth", "1"] + (["--branch", REPO_BRANCH] if REPO_BRANCH else [])
            if subprocess.run(cmd + [REPO_URL, REPO_DIR], env=quiet).returncode != 0:
                token = github_token()
                if not token:
                    raise SystemExit("git clone failed. Is the repository public? For a private one, "
                                     "add a Colab secret named GITHUB_TOKEN (in browser Colab: VS Code's "
                                     "extension can't read Colab secrets).")
                authed = REPO_URL.replace("https://", "https://x-access-token:" + token + "@")
                # capture_output: un mensaje de error no tiene que imprimir el token en el notebook
                if subprocess.run(cmd + [authed, REPO_DIR], env=quiet, capture_output=True).returncode != 0:
                    raise SystemExit("git clone failed even with GITHUB_TOKEN: check that the token "
                                     "can read this repository (fine-grained: Contents: Read).")
                # El token NO se guarda en .git/config: un pull posterior lo vuelve a pasar (abajo).
                subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", REPO_URL])
        os.chdir(REPO_DIR)
    # Traer los arreglos publicados después de tu primer clon (no hace nada si no hay).
    pulled = subprocess.run(["git", "pull", "--ff-only", "-q"], capture_output=True, text=True, env=quiet)
    token = github_token() if pulled.returncode != 0 else None
    if token:   # un repositorio privado: volver a hacer pull con el token, sin guardarlo
        branch = subprocess.run(["git", "rev-parse", "--abbrev-ref", "HEAD"], capture_output=True,
                                text=True).stdout.strip()
        authed = REPO_URL.replace("https://", "https://x-access-token:" + token + "@")
        pulled = subprocess.run(["git", "pull", "--ff-only", "-q", authed, branch], capture_output=True,
                                text=True, env=quiet)
    if pulled.returncode != 0:
        message = (pulled.stderr or pulled.stdout).strip()
        print("note: git pull skipped:", (message.replace(token, "***") if token else message)[:200])
    if os.getcwd() not in sys.path:
        sys.path.insert(0, os.getcwd())

    # Instalar una vez por VM: una marca con clave en el contenido de los archivos de requirements.
    reqs = [f for f in ("requirements.txt", "requirements-group.txt") if os.path.isfile(f)]
    digest = hashlib.sha256(b"".join(open(f, "rb").read() for f in reqs)).hexdigest()[:12]
    stamp = "/content/.lab_pip_" + digest
    if not os.path.exists(stamp):
        pip = [sys.executable, "-m", "pip", "install", "-q"]
        for f in reqs:
            pip += ["-r", f]
        subprocess.run(pip, check=True)  # UNA sola resolución: una versión del grupo que choca con una del núcleo falla ruidosamente
        open(stamp, "w").close()

    if MODELS_ON_DRIVE and not USE_DRIVE:
        print("note: MODELS_ON_DRIVE needs USE_DRIVE = True; the model cache stays on the VM disk.")
    if USE_DRIVE:
        # ANTES de `import environment`: un directorio artifacts/ real creado primero le ganaría
        # en silencio al link, y todo terminaría en el disco efímero.
        if not os.path.isdir("/content/drive/MyDrive"):   # VS Code: ya montado por su comando
            # Se avisa ANTES de la llamada: en la extensión de Colab de VS Code, drive.mount() puede colgarse en vez de fallar.
            print("Mounting Drive... (in VS Code, if this hangs: interrupt, run 'Colab: Mount Google Drive "
                  "to Server...' from Ctrl+Shift+P, then re-run this cell)")
            try:
                from google.colab import drive
                drive.mount("/content/drive")
            except Exception as e:  # drive.mount() desde una celda no funciona en la extensión de Colab de VS Code
                raise SystemExit("Drive is not mounted (" + type(e).__name__ + "). In VS Code: Ctrl+Shift+P > "
                                 "'Colab: Mount Google Drive to Server...', then re-run this cell.")
        os.makedirs(DRIVE + "/artifacts", exist_ok=True)
        if os.path.islink("artifacts"):
            print("artifacts ->", os.readlink("artifacts"))
        elif os.path.isdir("artifacts"):
            print("WARNING: artifacts/ is already a real directory, so it is NOT on Drive.")
            print("Move its contents into", DRIVE + "/artifacts", ", delete it, then re-run this cell.")
        else:
            os.symlink(DRIVE + "/artifacts", "artifacts")
            print("artifacts ->", DRIVE + "/artifacts")
        os.environ["LAB_USE_DRIVE"] = "1"
        if MODELS_ON_DRIVE:
            os.makedirs(DRIVE + "/hf_cache", exist_ok=True)
            os.environ.setdefault("HF_HOME", DRIVE + "/hf_cache")

    print("Colab bootstrap OK. Working directory:", os.getcwd())

## 1. Entorno y decisiones fijadas antes de medir

Usamos 320 tokens por hijo, incluidos doc_id y título de sección, en el tokenizer del reranker.
El solapamiento objetivo es 15%; no se cruzan secciones ni se inventan padres.
El contexto entregado ocupa como máximo 2.500 tokens de Qwen3, incluidos IDs y separadores.
Este presupuesto es solo de pasajes: la etapa 4 deberá contar el prompt completo y reservar salida.

BM25 usa stemming español y la misma normalización léxica del diagnóstico de etapa 2.
El cambio de secciones completas a chunks impide interpretar el diagnóstico anterior como una fila pareada.
No se ajustan umbrales de abstención con los puntajes de las preguntas fuera del corpus.

In [ ]:
import environment  # primero: mismas guardas del proyecto
from pathlib import Path
import importlib.util
import json
import random
import sys
import warnings

needed = [name for name in ['bm25s', 'Stemmer', 'transformers', 'huggingface_hub']
          if importlib.util.find_spec(name) is None]
if needed:
    raise RuntimeError(
        f'Faltan dependencias: {needed}. Instalá requirements-stage3.txt o ejecutá '
        '%pip install bm25s==0.2.14 PyStemmer==3.1.0 transformers==4.57.6 y reiniciá el kernel.'
    )

SMOKE_TEST = True
EVAL_SPLIT = 'dev'  # cambiar a test solo cuando las decisiones estén fijadas
SEED = 1337
DOCUMENTS_PATH = environment.artifacts_dir(False) / 'documents.jsonl'
EVAL_DIR = None  # opcional: Path('/ruta/a/eval'); nunca usar eval_draft
ART = environment.artifacts_dir(SMOKE_TEST)
PROFILE = 'smoke' if SMOKE_TEST else 'full'

if EVAL_SPLIT not in ('dev', 'test'):
    raise ValueError('Elegí dev o test explícitamente; no mezclar ambos en una tabla.')
if SMOKE_TEST and EVAL_SPLIT != 'dev':
    raise ValueError('La corrida smoke usa dev; test se reserva para una corrida full.')
environment.summary()

## 2. Funciones de recuperación y medición

Esta celda es una copia del módulo local, no depende de haberlo subido al repositorio remoto.
El hash del código forma parte de la procedencia de la corrida.

In [ ]:
"""Etapa 3 inicial: BM25, chunks estructurales y entrega por tokens.

El notebook incluye una copia de este módulo para poder abrirse solo en Colab.
No llama al LLM ni modifica el conjunto de evaluación congelado.
"""
from __future__ import annotations

from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
import hashlib
import importlib.metadata
import json
import math
import random
import re
import time
import unicodedata


STOP = set(
    "de la que el en y los se del las un por con no una su para "
    "es al lo como más a o sus le les".split()
)
EVAL_FILES = ("single_turn.jsonl", "dialogues.jsonl", "scenarios.jsonl")


def canonical_hash(value):
    blob = json.dumps(value, sort_keys=True, ensure_ascii=False, separators=(",", ":"))
    return hashlib.sha256(blob.encode("utf-8")).hexdigest()


def file_hash(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def read_jsonl(path):
    return [json.loads(line) for line in Path(path).read_text(encoding="utf-8").splitlines() if line.strip()]


def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")


def write_jsonl(path, rows):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text("".join(json.dumps(r, ensure_ascii=False) + "\n" for r in rows), encoding="utf-8")


def verify_frozen_files(eval_dir):
    """Verifica los tres conjuntos, sin escribir ni volver a congelar."""
    eval_dir = Path(eval_dir)
    frozen = json.loads((eval_dir / "FROZEN.json").read_text(encoding="utf-8"))
    for name in EVAL_FILES:
        rows = read_jsonl(eval_dir / name)
        spec = frozen["files"][name]
        if len(rows) != spec["rows"] or canonical_hash(rows) != spec["sha256_canonical"]:
            raise ValueError(f"Evaluación modificada: {name}. No se recalculó FROZEN.json.")
    return frozen


@dataclass(frozen=True)
class RetrievalConfig:
    name: str = "bm25"
    child_tokens: int = 320
    overlap_fraction: float = 0.15
    budget_tokens: int = 2500
    candidate_k: int = 100
    metric_ks: tuple = (1, 3, 5)
    bm25_k1: float = 1.5
    bm25_b: float = 0.75
    seed: int = 1337
    bootstrap_samples: int = 2000

    def __post_init__(self):
        if self.name != "bm25":
            raise ValueError("Esta versión implementa solo BM25. No se simulan otros brazos.")
        if self.child_tokens < 32 or not 0 <= self.overlap_fraction < 0.5:
            raise ValueError("child_tokens >= 32 y overlap_fraction en [0, 0.5).")
        if self.budget_tokens < 1 or self.candidate_k < 1:
            raise ValueError("Presupuesto y candidatos deben ser positivos.")
        if not self.metric_ks or min(self.metric_ks) < 1 or max(self.metric_ks) > self.candidate_k:
            raise ValueError("metric_ks debe estar entre 1 y candidate_k.")
        if self.bootstrap_samples < 1 or self.bm25_k1 <= 0 or not 0 <= self.bm25_b <= 1:
            raise ValueError("Parámetros BM25 o bootstrap inválidos.")


def token_count(tokenizer, text):
    return len(tokenizer.encode(text, add_special_tokens=False))


def chunk_documents(documents, tokenizer, cfg):
    """Offsets del texto original; nunca mezcla secciones ni decodifica tokens cortados.

    Dimensiona en el tokenizer del reranker, prefijo incluido. La entrega se cuenta
    aparte en el tokenizer de Qwen. No carga los pesos de ningún modelo.
    """
    chunks = []
    doc_ids = set()
    for doc in sorted(documents, key=lambda d: d["doc_id"]):
        doc_id = doc["doc_id"]
        if doc_id in doc_ids:
            raise ValueError(f"doc_id duplicado: {doc_id}")
        doc_ids.add(doc_id)
        metadata = doc.get("metadata", {})
        seen_sections = set()
        for sec in sorted(doc["sections"], key=lambda s: s["start"]):
            sec_id = sec["section_id"]
            if sec_id in seen_sections:
                raise ValueError(f"section_id duplicado: {sec_id}")
            seen_sections.add(sec_id)
            if not 0 <= sec["start"] < sec["end"] <= len(doc["text"]):
                raise ValueError(f"Offsets inválidos: {sec_id}")
            body = doc["text"][sec["start"]:sec["end"]]
            if not body.strip():
                continue
            # Prefijo mínimo fijo: identificador normativo y título de la sección.
            # No recorta títulos o texto jurídico en silencio.
            prefix = f"{doc_id} | {sec['title']}\n"
            prefix_n = token_count(tokenizer, prefix)
            room = cfg.child_tokens - prefix_n
            if room < 16:
                raise ValueError(f"Prefijo demasiado largo para el chunk: {sec_id}")
            encoded = tokenizer(body, add_special_tokens=False, return_offsets_mapping=True, truncation=False)
            offsets = encoded["offset_mapping"]
            if not offsets:
                raise ValueError(f"Texto no vacío sin tokens: {sec_id}")
            start_i = 0
            while start_i < len(offsets):
                end_i = min(start_i + room, len(offsets))
                start_char = 0 if start_i == 0 else offsets[start_i][0]
                end_char = len(body) if end_i == len(offsets) else offsets[end_i][0]
                text = body[start_char:end_char]
                # Retokenizar el fragmento aislado puede cambiar sus bordes BPE.
                while token_count(tokenizer, prefix + text) > cfg.child_tokens:
                    end_i -= 1
                    if end_i <= start_i:
                        raise ValueError(f"No se pudo dimensionar un fragmento: {sec_id}")
                    end_char = offsets[end_i][0]
                    text = body[start_char:end_char]
                if end_char <= start_char:
                    raise ValueError(f"Fragmento sin avance: {sec_id}")
                if text.strip():
                    absolute_start, absolute_end = sec["start"] + start_char, sec["start"] + end_char
                    identity = {"section_id": sec_id, "start": absolute_start, "end": absolute_end, "text": text}
                    chunks.append({
                        "chunk_id": f"{sec_id}@{absolute_start}:{absolute_end}-{canonical_hash(identity)[:12]}",
                        "doc_id": doc_id, "section_id": sec_id,
                        "title": doc["title"], "section_title": sec["title"],
                        "start": absolute_start, "end": absolute_end,
                        "text": text, "prefix": prefix, "index_text": prefix + text,
                        "reranker_tokens": token_count(tokenizer, prefix + text),
                        "metadata": metadata,
                    })
                if end_i == len(offsets):
                    break
                overlap = int((end_i - start_i) * cfg.overlap_fraction)
                start_i = max(start_i + 1, end_i - overlap)
    if not chunks:
        raise ValueError("El corpus no produjo chunks.")
    if len({c["chunk_id"] for c in chunks}) != len(chunks):
        raise ValueError("IDs de chunk duplicados.")
    return chunks


def validate_gold_in_index(items, chunks):
    available = {(c["doc_id"], c["section_id"]) for c in chunks}
    missing = []
    for item in items:
        if item["answerable"] and not item["gold"]:
            raise ValueError(f"Ítem respondible sin gold: {item['item_id']}")
        for gold in item["gold"]:
            pair = (gold["doc_id"], gold["section_id"])
            if pair not in available:
                missing.append((item["item_id"], pair))
    if missing:
        raise ValueError(f"Gold ausente del índice: {missing[:10]}")


class BM25Retriever:
    def __init__(self, chunks, cfg):
        import bm25s
        import Stemmer
        self.bm25s = bm25s
        self.stemmer = Stemmer.Stemmer("spanish")
        self.cfg = cfg
        self.chunks = chunks
        texts = [" ".join(self.words(c["index_text"])) for c in chunks]
        tokens = self.tokenize(texts)
        self.index = bm25s.BM25(k1=cfg.bm25_k1, b=cfg.bm25_b, method="lucene")
        self.index.index(tokens, show_progress=False)

    def words(self, text):
        text = unicodedata.normalize("NFC", text.lower())
        text = re.sub(r"\b(\d{2})\.(\d{3})\b", r"\1\2", text)
        return [self.stemmer.stemWord(w) for w in re.findall(r"\w+", text) if w not in STOP]

    def tokenize(self, texts):
        return self.bm25s.tokenize(texts, stopwords=[], token_pattern=r"(?u)\b\w+\b", show_progress=False)

    def retrieve(self, query):
        words = self.words(query)
        if not words:
            return []
        indices, scores = self.index.retrieve(
            self.tokenize([" ".join(words)]),
            k=min(self.cfg.candidate_k, len(self.chunks)), show_progress=False,
        )
        ranked = [
            {**self.chunks[int(i)], "score": float(score)}
            for i, score in zip(indices[0], scores[0]) if float(score) > 0
        ]
        return sorted(ranked, key=lambda c: (-c["score"], c["chunk_id"]))


def passage_block(chunk):
    # Este mismo bloque se entrega a la etapa 4, incluidos sus identificadores.
    return f"[{chunk['doc_id']} · {chunk['section_id']} · {chunk['chunk_id']}]\n{chunk['index_text']}"


def render_context(chunks):
    return "\n\n".join(passage_block(c) for c in chunks)


def deliver_by_budget(ranked, tokenizer, budget):
    delivered, skipped = [], []
    for c in ranked:
        if token_count(tokenizer, render_context(delivered + [c])) <= budget:
            delivered.append(c)
        else:
            skipped.append(c["chunk_id"])
    used = token_count(tokenizer, render_context(delivered))
    if used > budget:
        raise AssertionError("La entrega excedió el presupuesto.")
    return delivered, skipped, used


def normalize_answer(text):
    text = unicodedata.normalize("NFD", text.lower())
    text = "".join(c for c in text if unicodedata.category(c) != "Mn")
    text = re.sub(r"\(\s*\d+\s*\)", " ", text)
    text = re.sub(r"(?<=\d)[.,\s](?=\d{3}(?:\D|$))", "", text)
    text = re.sub(r"[^\w\s]", " ", text)
    return " ".join(text.split())


def contains_key(text, keys):
    normalized = f" {normalize_answer(text)} "
    return any(
        bool(normalize_answer(key)) and f" {normalize_answer(key)} " in normalized
        for key in keys or []
    )


def score_retrieval(item, ranked, delivered, ks):
    """k = posiciones de chunks recuperados; MRR truncado a candidate_k.

    Fuera del corpus: métricas de recuperación nulas, nunca fallos artificiales.
    answer_delivered busca claves SOLO en cuerpos, no en títulos/IDs del prefijo.
    """
    metrics = {"max_score": ranked[0]["score"] if ranked else 0.0}
    gold = {(g["doc_id"], g["section_id"]) for g in item["gold"]}
    gold_docs = {g[0] for g in gold}
    answerable = item["answerable"]
    for k in ks:
        top = ranked[:k]
        metrics[f"doc_hit@{k}"] = int(any(c["doc_id"] in gold_docs for c in top)) if answerable else None
        metrics[f"section_hit@{k}"] = int(any((c["doc_id"], c["section_id"]) in gold for c in top)) if answerable else None
    for level in ["doc", "section"]:
        position = next((i for i, c in enumerate(ranked, 1) if (
            c["doc_id"] in gold_docs if level == "doc" else (c["doc_id"], c["section_id"]) in gold
        )), None)
        metrics[f"{level}_mrr_candidates"] = (1.0 / position if position else 0.0) if answerable else None
    metrics["delivered_section_hit"] = int(any((c["doc_id"], c["section_id"]) in gold for c in delivered)) if answerable else None
    keys = item.get("answer_key")
    metrics["answer_delivered"] = int(any(contains_key(c["text"], keys) for c in delivered)) if answerable and keys else None
    return metrics


def reranker_overflow(query, candidates, tokenizer, max_length=512):
    lengths = [len(tokenizer(query, c["index_text"], add_special_tokens=True, truncation=False)["input_ids"]) for c in candidates]
    return {"pairs": len(lengths), "over_limit": sum(n > max_length for n in lengths), "max_pair_tokens": max(lengths, default=0)}


def evaluate_items(items, retriever, generator_tokenizer, reranker_tokenizer, run_spec):
    run_key = canonical_hash(run_spec)
    rows = []
    cfg = retriever.cfg
    for item in items:
        t0 = time.perf_counter()
        ranked = retriever.retrieve(item["question"])
        search_s = time.perf_counter() - t0
        t1 = time.perf_counter()
        delivered, skipped, used = deliver_by_budget(ranked, generator_tokenizer, cfg.budget_tokens)
        packing_s = time.perf_counter() - t1
        rows.append({
            "run_key": run_key, "run_spec": run_spec,
            "item_id": item["item_id"], "question": item["question"],
            "category": item["category"], "split": item["split"],
            "answerable": item["answerable"], "gold": item["gold"],
            "answer_key": item.get("answer_key"), "expected_verdict": item.get("expected_verdict"),
            "config": asdict(cfg),
            "metrics": score_retrieval(item, ranked, delivered, cfg.metric_ks),
            "tokens_delivered": used, "budget_tokens": cfg.budget_tokens,
            "search_s": search_s, "packing_s": packing_s, "retrieval_s": search_s + packing_s,
            "reranker_window_check": reranker_overflow(item["question"], ranked, reranker_tokenizer),
            "candidates": [{k: c[k] for k in ["chunk_id", "doc_id", "section_id", "score"]} for c in ranked],
            "delivered": [{**c, "passage": passage_block(c)} for c in delivered],
            "skipped_chunk_ids": skipped,
        })
    return rows


def bootstrap_mean(values, seed=1337, samples=2000):
    if not values:
        return None, None
    rng = random.Random(seed)
    means = sorted(sum(rng.choices(values, k=len(values))) / len(values) for _ in range(samples))
    return means[int(0.025 * (samples - 1))], means[int(0.975 * (samples - 1))]


def summary_rows(rows, cfg):
    result = []
    for category in ["all"] + sorted({r["category"] for r in rows}):
        group = [r for r in rows if category == "all" or r["category"] == category]
        if not group:
            continue
        for metric in rows[0]["metrics"]:
            if metric == "max_score":
                continue  # Los puntajes OOC se inspeccionan aparte; no son exactitud.
            values = [r["metrics"][metric] for r in group if r["metrics"][metric] is not None]
            low, high = bootstrap_mean(values, cfg.seed, cfg.bootstrap_samples)
            result.append({"category": category, "metric": metric, "n": len(values),
                           "mean": sum(values) / len(values) if values else None,
                           "ci95_low": low, "ci95_high": high})
    return result


def paired_comparison(previous, current, metric="answer_delivered"):
    """Para futuras filas: mismos ítems, split, benchmark y presupuesto."""
    a, b = {r["item_id"]: r for r in previous}, {r["item_id"]: r for r in current}
    if len(a) != len(previous) or len(b) != len(current) or a.keys() != b.keys():
        raise ValueError("La comparación requiere los mismos ítems, sin duplicados.")
    wins = losses = ties = 0
    differences = []
    for ident in sorted(a):
        x, y = a[ident], b[ident]
        if (x["split"] != y["split"] or x["budget_tokens"] != y["budget_tokens"]
                or x["run_spec"]["eval"] != y["run_spec"]["eval"]
                or x["run_spec"]["corpus_hash"] != y["run_spec"]["corpus_hash"]):
            raise ValueError("Evaluación, corpus, split o presupuesto diferentes.")
        p, q = x["metrics"][metric], y["metrics"][metric]
        if p is None or q is None:
            continue
        if p not in (0, 1) or q not in (0, 1):
            raise ValueError("El test de signos requiere una métrica binaria.")
        wins += q > p
        losses += q < p
        ties += q == p
        differences.append(q - p)
    n = wins + losses
    p_value = min(1.0, 2 * sum(math.comb(n, i) for i in range(min(wins, losses) + 1)) / 2**n) if n else 1.0
    low, high = bootstrap_mean(differences)
    return {"metric": metric, "wins": wins, "losses": losses, "ties": ties,
            "sign_test_p_two_sided": p_value, "delta_ci95": [low, high]}


def package_versions():
    versions = {}
    for name in ["bm25s", "PyStemmer", "transformers", "tokenizers", "numpy", "scipy", "huggingface-hub"]:
        try:
            versions[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            versions[name] = None
    return versions


def project_sources():
    """Huellas de los auxiliares cargados, además del código de esta etapa."""
    import sys
    sources = {}
    for name in ["environment", "schema", "console", "store", "llm"]:
        module = sys.modules.get(name)
        path = getattr(module, "__file__", None)
        if path and Path(path).is_file():
            sources[name] = {"sha256": file_hash(path)}
    return sources


def save_index(retriever, index_dir, manifest):
    index_dir = Path(index_dir)
    index_dir.mkdir(parents=True, exist_ok=True)
    existing = index_dir / "manifest.json"
    if existing.exists() and canonical_hash(json.loads(existing.read_text(encoding="utf-8"))) != canonical_hash(manifest):
        raise ValueError("El directorio de índice pertenece a otra configuración.")
    retriever.index.save(str(index_dir / "bm25"))
    write_jsonl(index_dir / "chunks.jsonl", retriever.chunks)
    write_json(existing, manifest)


def utc_now():
    return datetime.now(timezone.utc).isoformat()

In [ ]:
IMPLEMENTATION_SHA256 = 'e764fae03353a924d67219dee52b990f52e25756ddad1ad6bf72074e6180ebf6'
CFG = RetrievalConfig(seed=SEED)
print(asdict(CFG))

## 3. Validar corpus y congelamiento

Se verifica la evaluación completa contra el corpus completo antes de seleccionar dev o smoke.
Si hay varias copias distintas del conjunto congelado, se detiene para que elijas `EVAL_DIR`.
Si faltan insumos, ejecutar etapa 1 y traer la carpeta congelada; no reconstruir etiquetas aquí.

In [ ]:
import schema
from schema import Document

if not DOCUMENTS_PATH.is_file():
    raise FileNotFoundError(
        f'Falta {DOCUMENTS_PATH}. Ejecutá 01_documents.ipynb en el mismo Drive '
        'o recuperá documents.jsonl; no se genera un corpus alternativo.'
    )

if EVAL_DIR is None:
    options = [Path('eval'), Path('etapa2_congelada/eval'),
               DOCUMENTS_PATH.parent / 'etapa2_congelada/eval']
    existing = list(dict.fromkeys(p.resolve() for p in options if (p / 'FROZEN.json').is_file()))
    if not existing:
        raise FileNotFoundError(
            'Falta la evaluación congelada. Subí etapa2_congelada/eval al proyecto '
            'o fijá EVAL_DIR a la carpeta que contiene FROZEN.json y los tres JSONL.'
        )
    versions = {canonical_hash(verify_frozen_files(p)) for p in existing}
    if len(versions) > 1:
        raise RuntimeError(f'Hay evaluaciones congeladas diferentes: {existing}. Elegí EVAL_DIR.')
    EVAL_DIR = existing[0]
EVAL_DIR = Path(EVAL_DIR)
FROZEN = verify_frozen_files(EVAL_DIR)
schema.validate_documents(DOCUMENTS_PATH).show().raise_if_failed()
schema.load_eval(EVAL_DIR, documents_path=DOCUMENTS_PATH)
schema.validate_eval(EVAL_DIR, DOCUMENTS_PATH).show().raise_if_failed()

all_documents = read_jsonl(DOCUMENTS_PATH)
corpus_hash = schema.documents_hash([Document.model_validate(d) for d in all_documents])
if corpus_hash != FROZEN['documents_sha256']:
    raise ValueError('El hash del corpus difiere del usado para congelar la evaluación.')
all_items = read_jsonl(EVAL_DIR / 'single_turn.jsonl')
items = [q for q in all_items if q['split'] == EVAL_SPLIT]
if not items:
    raise ValueError(f'No hay preguntas para {EVAL_SPLIT}.')
documents = all_documents

if SMOKE_TEST:
    # Hasta dos preguntas por categoría; todos los gold y diez distractores con semilla.
    selected = []
    for category in sorted({q['category'] for q in items}):
        selected.extend([q for q in items if q['category'] == category][:2])
    items = selected
    gold_ids = {g['doc_id'] for q in items for g in q['gold']}
    others = sorted([d['doc_id'] for d in all_documents if d['doc_id'] not in gold_ids])
    sampled = set(random.Random(SEED).sample(others, min(10, len(others))))
    documents = [d for d in all_documents if d['doc_id'] in gold_ids | sampled]

pending = sum('pendiente' in q.get('notes', '').lower() for q in items)
if pending:
    warnings.warn(f'{pending}/{len(items)} preguntas aún declaran revisión humana pendiente. '
                  'Los resultados deben considerarse provisionales.')
print('Perfil:', PROFILE, '| split:', EVAL_SPLIT)
print('Evaluación:', EVAL_DIR)
print('Documentos indexados:', len(documents), '/', len(all_documents))
print('Preguntas seleccionadas:', len(items))
print('Hash del corpus completo:', corpus_hash)

## 4. Tokenizers abiertos con revisión fijada

Se utiliza el tokenizer de `cross-encoder/mmarco-mMiniLMv2-L12-H384-v1` para dimensionar
hijos y comprobar pares de 512 tokens; no se ejecuta un reranker en esta línea de base.
El tokenizer de `Qwen/Qwen3-8B` cuenta la entrega. No se descargan pesos de los modelos.
La primera ejecución resuelve las revisiones y las guarda; las siguientes reutilizan esos commits.

In [ ]:
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer

tokenizer_models = {
    'generator': 'Qwen/Qwen3-8B',
    'reranker': 'cross-encoder/mmarco-mMiniLMv2-L12-H384-v1',
}
lock_path = ART / 'index' / 'stage3_tokenizer_revisions.json'
locks = json.loads(lock_path.read_text(encoding='utf-8')) if lock_path.exists() else {}
tokenizers = {}
for role, model in tokenizer_models.items():
    if role in locks and locks[role]['model'] != model:
        raise ValueError('El lock de tokenizers pertenece a otros modelos.')
    revision = locks.get(role, {}).get('revision', 'main')
    snapshot = Path(snapshot_download(
        repo_id=model, revision=revision,
        allow_patterns=['tokenizer.json', 'tokenizer_config.json', 'special_tokens_map.json',
                        'vocab.txt', 'vocab.json', 'merges.txt', 'config.json', 'added_tokens.json'],
    ))
    resolved = snapshot.name
    if not re.fullmatch(r'[0-9a-f]{40}', resolved):
        raise RuntimeError(f'No se pudo identificar el commit del tokenizer: {snapshot}')
    tokenizers[role] = AutoTokenizer.from_pretrained(snapshot, local_files_only=True, use_fast=True)
    locks[role] = {'model': model, 'revision': resolved}
write_json(lock_path, locks)
GEN_TOKENIZER = tokenizers['generator']
RERANK_TOKENIZER = tokenizers['reranker']
print(json.dumps(locks, indent=2))

## 5. Construir índice y comprobar gold

El manifiesto identifica corpus, documentos efectivamente indexados, tokenizers, configuración,
normalización y código. Cada cambio obtiene su propio directorio: no se reutiliza un índice incompatible.
La construcción BM25 se repite al ejecutar esta celda; se guarda el índice para las etapas siguientes.

In [ ]:
from console import StageRun

run = StageRun.start('03_retrieval', ART, profile=PROFILE, config=asdict(CFG))
started_at = utc_now()
build_start = time.perf_counter()
chunks = chunk_documents(documents, RERANK_TOKENIZER, CFG)
validate_gold_in_index(items, chunks)
retriever = BM25Retriever(chunks, CFG)
build_s = time.perf_counter() - build_start

index_manifest = {
    'stage': '03_retrieval', 'profile': PROFILE,
    'corpus_hash': corpus_hash,
    'indexed_documents_hash': canonical_hash(sorted(documents, key=lambda d: d['doc_id'])),
    'documents': len(documents), 'chunks': len(chunks),
    'chunks_hash': canonical_hash(chunks),
    'config': asdict(CFG), 'tokenizers': locks,
    'lexical': {'method': 'lucene', 'stemmer': 'spanish', 'stopwords': sorted(STOP),
                'identifier_normalization': '24.240 -> 24240',
                'prefix': 'doc_id | section_title'},
    'implementation_sha256': IMPLEMENTATION_SHA256, 'packages': package_versions(),
    'project_sources': project_sources(),
}
INDEX_KEY = canonical_hash(index_manifest)
INDEX_DIR = ART / 'index' / f'stage3_{INDEX_KEY[:16]}'
save_index(retriever, INDEX_DIR, index_manifest)
RUN_SPEC = {
    'stage': '03_retrieval', 'profile': PROFILE, 'split': EVAL_SPLIT,
    'config': asdict(CFG), 'eval': FROZEN, 'corpus_hash': corpus_hash,
    'index_key': INDEX_KEY, 'item_ids': [q['item_id'] for q in items],
    'implementation_sha256': IMPLEMENTATION_SHA256,
}
RUN_KEY = canonical_hash(RUN_SPEC)
RESULTS_DIR = ART / 'results' / f'stage3_{RUN_KEY[:16]}'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print('Chunks:', len(chunks), '| máximo:', max(c['reranker_tokens'] for c in chunks))
print('Construcción (s):', round(build_s, 2))
print('Índice:', INDEX_DIR)

## 6. Guardas que deben fallar antes de medir

Una referencia gold inexistente debe detener la evaluación. Un pasaje que no entra debe saltarse,
sin impedir entregar otro más pequeño que venga después. Estas pruebas no modifican los insumos.

In [ ]:
try:
    validate_gold_in_index([{
        'item_id': 'smoke_missing_gold', 'answerable': True,
        'gold': [{'doc_id': '__missing__', 'section_id': '__missing__#ART-1'}]
    }], chunks)
except ValueError:
    print('OK: un gold ausente se rechaza.')
else:
    raise AssertionError('No saltó la guarda de gold.')

small = {**chunks[0], 'chunk_id': 'guard_small', 'text': 'dato breve', 'index_text': 'dato breve'}
large = {**small, 'chunk_id': 'guard_large', 'text': 'dato largo ' * 100,
         'index_text': 'dato largo ' * 100}
guard_budget = token_count(GEN_TOKENIZER, render_context([small]))
kept, skipped, used = deliver_by_budget([large, small], GEN_TOKENIZER, guard_budget)
assert [c['chunk_id'] for c in kept] == ['guard_small'] and 'guard_large' in skipped
assert used <= guard_budget
print('OK: se salta un pasaje demasiado largo y se considera el siguiente.')

## 7. Medir y guardar resultados

`hit@k` usa las primeras k posiciones de chunks. MRR se trunca a los candidatos recuperados.
`delivered_section_hit` evalúa lo que realmente entró. `answer_delivered` es una aproximación léxica:
busca alguna clave en el cuerpo de un pasaje entregado, sin contar los identificadores del prefijo.
No certifica una respuesta correcta ni la vigencia jurídica. Fuera del corpus, estas métricas son nulas.

El contexto exacto queda en `contexts.jsonl`, listo para el ensamblador de etapa 4. El límite se aplica al
bloque completo de pasajes, sin cortar un fragmento para hacerlo entrar. Se consideran hasta 100 candidatos;
se registra el presupuesto realmente usado, aunque la lista no lo llene.

In [ ]:
rows = evaluate_items(items, retriever, GEN_TOKENIZER, RERANK_TOKENIZER, RUN_SPEC)
summary = summary_rows(rows, CFG)
write_jsonl(RESULTS_DIR / 'retrieval.jsonl', rows)
write_jsonl(RESULTS_DIR / 'summary.jsonl', summary)
write_json(RESULTS_DIR / 'run_spec.json', RUN_SPEC)
write_jsonl(RESULTS_DIR / 'contexts.jsonl', [
    {'run_key': RUN_KEY, 'item_id': r['item_id'], 'question': r['question'],
     'tokens_delivered': r['tokens_delivered'],
     'context': render_context(r['delivered']), 'passages': r['delivered']}
    for r in rows
])

import csv
with (RESULTS_DIR / 'summary.csv').open('w', encoding='utf-8', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=list(summary[0]))
    writer.writeheader()
    writer.writerows(summary)

try:
    import pandas as pd
except ImportError:
    for r in summary:
        print(r)
else:
    display(pd.DataFrame(summary))
print('Promedio de tokens entregados:', round(sum(r['tokens_delivered'] for r in rows) / len(rows), 1))
print('Resultados:', RESULTS_DIR)

## 8. Abrir la caja de los errores

Inspeccionar preguntas que no recibieron la sección o cuya clave no llegó, aunque se recuperó el documento.
No corregir gold automáticamente a partir de este ranking. Para test, documentar los fallos sin ajustar el pipeline.
El chequeo de ventana del reranker es preventivo: esta fila no lo ejecuta.

In [ ]:
for r in rows:
    if r['answerable'] and (not r['metrics']['delivered_section_hit'] or not r['metrics']['answer_delivered']):
        print('\n', r['item_id'], r['question'])
        print('Gold:', r['gold'])
        print('Métricas:', r['metrics'])
        print('Top 5:', [(c['section_id'], round(c['score'], 3)) for c in r['candidates'][:5]])
        print('Entregados:', [c['section_id'] for c in r['delivered']])

pair_count = sum(r['reranker_window_check']['pairs'] for r in rows)
overflow = sum(r['reranker_window_check']['over_limit'] for r in rows)
print('Pares consulta+hijo >512:', overflow, '/', pair_count)
print('Puntajes máximos; no equivalen a probabilidades de respuesta:')
for r in rows:
    print(r['item_id'], r['category'], r['metrics']['max_score'])

## 9. Cerrar la corrida y preparar el análisis

El bootstrap remuestrea preguntas y sus intervalos son orientativos con muestras pequeñas; se informan n
y categorías. Al comparar próximos brazos, usar victorias/derrotas/empates y test de signos pareado
(`paired_comparison`) con la misma evaluación y presupuesto. Con una sola fila no se afirma una mejora.

Completar después de ejecutar:

- ¿Cuántas veces acertó documento pero no sección? ¿Y sección entregada pero clave ausente?
- ¿Cuánto contexto consumió cada categoría? ¿El límite de 100 candidatos dejó presupuesto sin llenar?
- ¿Qué explica cada fallo: redacción, chunking, prefijo, gold o ranking?
- ¿Qué pieza probaríamos a continuación y qué hipótesis concreta pondría a prueba?

Pendiente para completar etapa 3: denso, RRF y las ablaciones que el grupo decida justificar.
No agregar padres, filtros o reranker sin implementar su comportamiento y registrar el cambio.

Referencias técnicas: [BM25S](https://github.com/xhluca/bm25s) y
[tokenizers de Transformers](https://huggingface.co/docs/transformers/main_classes/tokenizer).

In [ ]:
report = {
    'stage': '03_retrieval', 'status': 'ok', 'profile': PROFILE, 'split': EVAL_SPLIT,
    'started_at': started_at, 'finished_at': utc_now(), 'run_key': RUN_KEY,
    'run_spec': RUN_SPEC, 'index_dir': str(INDEX_DIR), 'results_dir': str(RESULTS_DIR),
    'documents': len(documents), 'chunks': len(chunks), 'items': len(rows),
    'index_build_s': build_s, 'llm_calls': 0,
    'mean_tokens_delivered': sum(r['tokens_delivered'] for r in rows) / len(rows),
    'reranker_pairs_over_limit': overflow, 'reranker_pairs_checked': pair_count,
    'human_review_pending_in_notes': pending,
    'limitations': ['BM25 only', 'smoke is not a benchmark' if SMOKE_TEST else 'single row; no improvement claim',
                    'answer_delivered is lexical containment; not semantic correctness'],
}
write_json(RESULTS_DIR / 'run_report.json', report)
run.note('documents', len(documents))
run.note('chunks', len(chunks))
run.note('items', len(rows))
run.finish()
print('Corrida finalizada:', RUN_KEY)
print('Para dev completo: SMOKE_TEST=False, EVAL_SPLIT="dev".')
print('Para reporte final, una vez fijadas las decisiones: SMOKE_TEST=False, EVAL_SPLIT="test".')